# Introduction

This notebook presents an end-to-end **credit card fraud detection** workflow: **binary classification** on transactional data with strong **class imbalance**.

**Objectives**
- Build reproducible pipelines (`StandardScaler` → **SMOTE** → classifier) to avoid data leakage.
- Tune a **Random Forest** with **GridSearchCV** using **F1-score** (focus on the minority fraud class).
- Compare several models, select the best on held-out data, and persist metrics, model, and figures.

**Target:** `Class` — `0` legitimate, `1` fraud.

**Environment:** Python 3.10+ recommended. Install dependencies with:

`pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib` — optional: `xgboost` (otherwise histogram gradient boosting from sklearn is used).

## Configuration & artifact paths

All mandatory outputs are saved in `ARTIFACT_DIR`. `GridSearchCV` uses a **stratified subsample** of the training set for speed (full training data is used for the final fitted models).

In [1]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

DATA_PATH = BASE_DIR / "creditcard.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path("creditcard.csv")
ARTIFACT_DIR = BASE_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
# Stratified subsample size for RF grid search (full train used for final refit)
GRIDSEARCH_TRAIN_SAMPLES = 25_000
CV_FOLDS = 2

FIG_KW = {"dpi": 150, "bbox_inches": "tight"}

print(f"Dataset path: {DATA_PATH.resolve()}")
print(f"Artifacts:      {ARTIFACT_DIR.resolve()}")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Place creditcard.csv in {BASE_DIR} or set DATA_PATH.")

Dataset path: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\creditcard.csv
Artifacts:      C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire


# Data Understanding

We load the CSV, inspect schema and basic statistics, quantify missingness, and save **class distribution** and **correlation heatmap** figures.

In [2]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from xgboost import XGBClassifier

    HAS_XGB = True
except ImportError:
    HAS_XGB = False

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)


def save_fig(path: Path, **kw) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(p, **{**FIG_KW, **kw})
    plt.close()
    print(f"Saved: {p.resolve()}")


def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    m = df.isnull().sum()
    pct = (m / len(df) * 100).round(4)
    return pd.DataFrame({"count": m, "pct": pct})


def plot_class_distribution(df: pd.DataFrame, target: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    vc = df[target].value_counts().sort_index()
    vc.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"], rot=0)
    ax.set_title("Class distribution")
    ax.set_xlabel("Class (0 = legitimate, 1 = fraud)")
    ax.set_ylabel("Count")
    save_fig(out_path)


def plot_correlation_heatmap(df: pd.DataFrame, out_path: Path, exclude=None):
    exclude = exclude or []
    num = df.select_dtypes(include=[np.number]).drop(columns=exclude, errors="ignore")
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(num.corr(), cmap="vlag", center=0, ax=ax, cbar_kws={"shrink": 0.5})
    ax.set_title("Feature correlation heatmap")
    save_fig(out_path)


def plot_feature_distributions(df: pd.DataFrame, cols: list, target: str, out_path: Path):
    cols = [c for c in cols if c in df.columns and c != target]
    n = len(cols)
    fig, axes = plt.subplots(n, 2, figsize=(11, 3 * max(n, 1)))
    if n == 1:
        axes = np.asarray([axes])
    for i, col in enumerate(cols):
        sns.histplot(df[col], bins=50, kde=True, ax=axes[i, 0], color="#3498db")
        axes[i, 0].set_title(f"Histogram: {col}")
        sns.boxplot(data=df, x=target, y=col, ax=axes[i, 1], palette=["#2ecc71", "#e74c3c"])
        axes[i, 1].set_title(f"{col} vs class")
    plt.tight_layout()
    save_fig(out_path)


def make_fraud_pipeline(classifier) -> ImbPipeline:
    return ImbPipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("clf", classifier),
        ]
    )


def stratified_train_subset(
    X: pd.DataFrame, y: pd.Series, n_samples: int, random_state: int
):
    n_samples = min(n_samples, len(X))
    if n_samples == len(X):
        return X, y
    X_sub, _, y_sub, _ = train_test_split(
        X,
        y,
        train_size=n_samples,
        stratify=y,
        random_state=random_state,
    )
    return X_sub, y_sub


def clf_params_from_grid(best_params: dict) -> dict:
    return {k.split("__", 1)[1]: v for k, v in best_params.items() if k.startswith("clf__")}


def evaluate_pipeline(pipe: ImbPipeline, X_test, y_test, name: str) -> dict:
    y_pred = pipe.predict(X_test)
    proba = None
    clf = pipe.named_steps["clf"]
    if hasattr(clf, "predict_proba"):
        proba = pipe.predict_proba(X_test)[:, 1]
    elif hasattr(clf, "decision_function"):
        proba = pipe.decision_function(X_test)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": np.nan if proba is None else roc_auc_score(y_test, proba),
    }


def plot_smote_effect(y_before, y_after, out_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    pd.Series(y_before).value_counts().sort_index().plot(
        kind="bar", ax=axes[0], color=["#3498db", "#e74c3c"], rot=0
    )
    axes[0].set_title("Training labels BEFORE SMOTE")
    axes[0].set_xlabel("Class")
    pd.Series(y_after).value_counts().sort_index().plot(
        kind="bar", ax=axes[1], color=["#3498db", "#e74c3c"], rot=0
    )
    axes[1].set_title("Training labels AFTER SMOTE (train only)")
    axes[1].set_xlabel("Class")
    plt.tight_layout()
    save_fig(out_path)

In [3]:
df = pd.read_csv(DATA_PATH)
TARGET = "Class"

if TARGET in df.columns:
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = df.dropna(subset=[TARGET]).reset_index(drop=True)
    df[TARGET] = df[TARGET].astype(int)

print(f"Shape: {df.shape[0]:,} × {df.shape[1]}")
display(df.head())
print()
df.info()
print()
display(df.describe())

Shape: 284,807 × 31


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0



<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-n

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.175161e-15,3.384974e-16,-1.379537e-15,2.094852e-15,1.021879e-15,1.494498e-15,-5.620335e-16,1.149614e-16,-2.414189e-15,...,1.628620e-16,-3.576577e-16,2.618565e-16,4.473914e-15,5.109395e-16,1.686100e-15,-3.661401e-16,-1.227452e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


In [4]:
miss = missing_value_report(df)
print("Missing values:")
display(miss[miss["count"] > 0] if miss["count"].any() else miss.head())
print("Total missing cells:", int(miss["count"].sum()))

plot_class_distribution(df, TARGET, ARTIFACT_DIR / "class_distribution.png")
plot_correlation_heatmap(df, ARTIFACT_DIR / "correlation_heatmap.png", exclude=[TARGET])

dist_cols = ["Time", "Amount", "V12", "V14", "V17"]
plot_feature_distributions(df, dist_cols, TARGET, ARTIFACT_DIR / "feature_distributions.png")

Missing values:


,count,pct
Time,0,0.0
V1,0,0.0
V2,0,0.0
V3,0,0.0
V4,0,0.0


Total missing cells: 0


Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\class_distribution.png


Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\correlation_heatmap.png


Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\feature_distributions.png


# Data Preparation

- Impute or drop missing values if present; keep feature matrix and target aligned.
- **No categoricals** in the standard `creditcard.csv`; one-hot encoding is applied only when string columns exist.
- **Train/test split (80/20), stratified** on fraud rate.
- **SMOTE** is illustrated on **training data only** (after scaling), matching the pipeline order; the modeling section uses the same logic inside `imblearn.pipeline.Pipeline`.

In [5]:
feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols].copy()
y = df[TARGET].copy()

if X.isnull().any().any():
    X = X.fillna(X.median(numeric_only=True))

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    print("Encoded:", cat_cols)
else:
    print("No categorical columns to encode.")

FEATURE_NAMES = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(
    f"Fraud rate train: {y_train.mean():.5f} | test: {y_test.mean():.5f}"
)

# SMOTE visualization (no leakage: train only)
_sc = StandardScaler().fit(X_train)
X_train_s = _sc.transform(X_train)
_sm = SMOTE(random_state=RANDOM_STATE)
_, y_smote_demo = _sm.fit_resample(X_train_s, y_train)
plot_smote_effect(y_train, y_smote_demo, ARTIFACT_DIR / "smote_class_distribution.png")

No categorical columns to encode.


Train: 227,845 | Test: 56,962
Fraud rate train: 0.00173 | test: 0.00172


Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\smote_class_distribution.png


# Modeling

Each estimator is wrapped in **`imblearn.pipeline.Pipeline`**: `StandardScaler` → `SMOTE` → classifier. Fitting applies scaling and SMOTE **only on training folds** inside `GridSearchCV`, and on full training data for final models.

**Random Forest:** `GridSearchCV` with stratified k-fold CV (`CV_FOLDS` in the config cell), **`scoring="f1"`**, on a stratified subset of `X_train` (for feasible runtime on laptops). Best hyperparameters are then used to refit a full pipeline on **all** of `X_train`.

In [6]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

X_tune, y_tune = stratified_train_subset(
    X_train, y_train, GRIDSEARCH_TRAIN_SAMPLES, RANDOM_STATE
)
print(
    f"GridSearch RF on {len(X_tune):,} stratified train rows (full train {len(X_train):,} for refit)."
)

rf_pipe_search = make_fraud_pipeline(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
)
param_grid_rf = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [25, None],
}

grid_rf = GridSearchCV(
    estimator=rf_pipe_search,
    param_grid=param_grid_rf,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)
grid_rf.fit(X_tune, y_tune)
print("Best RF params:", grid_rf.best_params_)
print("Best CV F1:", round(grid_rf.best_score_, 4))

rf_best_kwargs = clf_params_from_grid(grid_rf.best_params_)
pipe_rf = make_fraud_pipeline(
    RandomForestClassifier(**rf_best_kwargs, random_state=RANDOM_STATE, n_jobs=-1)
)
pipe_rf.fit(X_train, y_train)

pipe_lr = make_fraud_pipeline(
    LogisticRegression(max_iter=5000, solver="lbfgs", random_state=RANDOM_STATE)
)
pipe_lr.fit(X_train, y_train)

pipe_dt = make_fraud_pipeline(
    DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE)
)
pipe_dt.fit(X_train, y_train)

if HAS_XGB:
    gb_est = XGBClassifier(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        verbosity=0,
        n_jobs=-1,
    )
    gb_label = "Gradient Boosting (XGBoost)"
else:
    gb_est = HistGradientBoostingClassifier(
        max_iter=150,
        max_depth=7,
        learning_rate=0.08,
        random_state=RANDOM_STATE,
    )
    gb_label = "Gradient Boosting (HistGradientBoosting)"

pipe_gb = make_fraud_pipeline(gb_est)
pipe_gb.fit(X_train, y_train)

fitted_pipelines = {
    "Logistic Regression": pipe_lr,
    "Decision Tree": pipe_dt,
    "Random Forest (tuned)": pipe_rf,
    gb_label: pipe_gb,
}
print("Fitted models:", list(fitted_pipelines.keys()))

GridSearch RF on 25,000 stratified train rows (full train 227,845 for refit).
Fitting 2 folds for each of 4 candidates, totalling 8 fits


Best RF params: {'clf__max_depth': 25, 'clf__n_estimators': 100}
Best CV F1: 0.8057


Fitted models: ['Logistic Regression', 'Decision Tree', 'Random Forest (tuned)', 'Gradient Boosting (XGBoost)']


# Evaluation

Metrics are computed on the **held-out test set** (original imbalance — **not** SMOTEd). Each pipeline applies scaling and SMOTE only during `fit` on training data; `predict` on test uses scaling learned from train, **without** resampling test rows.

In [7]:
rows = [
    evaluate_pipeline(pipe, X_test, y_test, name)
    for name, pipe in fitted_pipelines.items()
]
results_df = pd.DataFrame(rows)

out_csv = ARTIFACT_DIR / "model_results.csv"
results_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")

print("\nRaw results:")
display(results_df.round(4))

sorted_f1 = results_df.sort_values(["F1", "ROC-AUC"], ascending=False).reset_index(drop=True)
sorted_auc = results_df.sort_values(["ROC-AUC", "F1"], ascending=False).reset_index(drop=True)
print("\nSorted by F1-score, then ROC-AUC:")
display(sorted_f1.round(4))
print("\nSorted by ROC-AUC, then F1:")
display(sorted_auc.round(4))

Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\model_results.csv

Raw results:


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.9741,0.0578,0.9184,0.1088,0.9708
1,Decision Tree,0.9865,0.0972,0.8265,0.1740,0.8689
2,Random Forest (tuned),0.9993,0.8041,0.7959,0.8000,0.9783
3,Gradient Boosting (XGBoost),0.9980,0.4521,0.8673,0.5944,0.9759



Sorted by F1-score, then ROC-AUC:


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Random Forest (tuned),0.9993,0.8041,0.7959,0.8000,0.9783
1,Gradient Boosting (XGBoost),0.9980,0.4521,0.8673,0.5944,0.9759
2,Decision Tree,0.9865,0.0972,0.8265,0.1740,0.8689
3,Logistic Regression,0.9741,0.0578,0.9184,0.1088,0.9708



Sorted by ROC-AUC, then F1:


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Random Forest (tuned),0.9993,0.8041,0.7959,0.8000,0.9783
1,Gradient Boosting (XGBoost),0.9980,0.4521,0.8673,0.5944,0.9759
2,Logistic Regression,0.9741,0.0578,0.9184,0.1088,0.9708
3,Decision Tree,0.9865,0.0972,0.8265,0.1740,0.8689


# Results

We select the best model using **test F1-score** (tie-break: **ROC-AUC**), consistent with the grid-search objective for Random Forest. The **entire fitted pipeline** (scaler + SMOTE settings captured in last fit + classifier) is persisted for deployment. In production you would typically export **only** `StandardScaler` + classifier trained on a chosen policy (SMOTE is a training-time strategy; operational pipelines often use threshold tuning or cost-sensitive learning instead of synthetic oversampling at scoring time).

In [8]:
best_name = sorted_f1.iloc[0]["Model"]
best_pipe = fitted_pipelines[best_name]

print("=" * 64)
print("BEST MODEL (test set): ", best_name)
print("=" * 64)
display(sorted_f1.head(1).round(4))

joblib.dump(best_pipe, ARTIFACT_DIR / "best_model.pkl")
print(f"Saved: {(ARTIFACT_DIR / 'best_model.pkl').resolve()}")

y_pred_best = best_pipe.predict(X_test)
y_proba_best = best_pipe.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best, ax=ax, cmap="Blues", colorbar=True
)
ax.set_title(f"Confusion matrix — {best_name}")
plt.tight_layout()
save_fig(ARTIFACT_DIR / "confusion_matrix.png")

fpr, tpr, _ = roc_curve(y_test, y_proba_best)
roc_auc_val = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, lw=2, label=f"ROC AUC = {roc_auc_val:.4f}")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title(f"ROC curve — {best_name}")
ax.legend(loc="lower right")
plt.tight_layout()
save_fig(ARTIFACT_DIR / "roc_curve.png")

clf = best_pipe.named_steps["clf"]
fig, ax = plt.subplots(figsize=(10, 7))
if hasattr(clf, "feature_importances_"):
    imp = clf.feature_importances_
elif hasattr(clf, "coef_"):
    imp = np.abs(clf.coef_).ravel()
else:
    imp = None

if imp is not None:
    names = np.array(FEATURE_NAMES)
    k = min(25, len(imp))
    order = np.argsort(imp)[::-1][:k]
    sns.barplot(x=imp[order], y=names[order], ax=ax, orient="h", color="#2980b9")
    ax.set_xlabel("Importance / |coefficient|")
else:
    ax.text(0.5, 0.5, "No feature_importances_ or coef_ available.", ha="center")
ax.set_title(f"Top features — {best_name}")
plt.tight_layout()
save_fig(ARTIFACT_DIR / "feature_importance.png")

print("\nMandatory artifacts written to:", ARTIFACT_DIR.resolve())

BEST MODEL (test set):  Random Forest (tuned)


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Random Forest (tuned),0.9993,0.8041,0.7959,0.8,0.9783


Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\best_model.pkl


Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\confusion_matrix.png


Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\roc_curve.png


Saved: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire\feature_importance.png

Mandatory artifacts written to: C:\Med\Polytech\Machine Learning\Sem4\ProjetFraudeBanquaire


## Business interpretation

### Why class imbalance matters

Legitimate transactions vastly outnumber fraud. A naive model that **always predicts “legitimate”** can still show **high accuracy** while failing completely at fraud detection. Business decisions must rely on metrics that reflect **minority-class performance**, not accuracy alone.

### Why F1-score is central here

**F1** is the harmonic mean of **precision** and **recall** on the fraud class. It rewards a **balance** between catching fraud (**recall**) and avoiding too many false alarms (**precision**). For Random Forest we tuned with **GridSearchCV** and **`scoring="f1"`** so hyperparameters favor this balance on cross-validation folds (on a training subsample for efficiency), then we refit on the full training set.

### Cost of false negatives (undetected fraud)

A **false negative** is a fraudulent transaction classified as legitimate. Consequences include **direct financial loss**, **customer trust** damage, regulatory scrutiny, and operational cost of investigations. In many settings **recall** on fraud is therefore prioritized, sometimes at the expense of precision — the right trade-off depends on **costs** (false negative vs false positive) defined with stakeholders.

### Why the selected model is preferred

The **best model** in this notebook is the one with the highest **test F1** (with **ROC-AUC** as tie-breaker). **ROC-AUC** summarizes how well the model **ranks** fraud vs non-fraud across thresholds; a higher value usually indicates better separability. The chosen model offers the strongest combined ranking and F1 on held-out data among the candidates, under the same preprocessing and SMOTE-in-pipeline training protocol. **Feature importance** (or coefficient magnitudes for linear models) highlights which inputs drive predictions and can guide monitoring and domain review — remembering that anonymized `V*` features are PCA-like and not as directly actionable as raw merchant attributes.

# Conclusion

This notebook loads `creditcard.csv`, explores imbalance and multicollinearity patterns, prepares data with stratified **80/20** splitting, and trains **imblearn pipelines** (`StandardScaler` → **SMOTE** → classifier) so **SMOTE never touches the test set**. **Random Forest** hyperparameters are tuned with **GridSearchCV** maximizing **F1**. All models are compared on **Accuracy, Precision, Recall, F1, and ROC-AUC**; results are saved to **`model_results.csv`**, the winning pipeline to **`best_model.pkl`**, and figures to **`class_distribution.png`**, **`correlation_heatmap.png`**, **`smote_class_distribution.png`**, **`feature_distributions.png`**, **`confusion_matrix.png`**, **`roc_curve.png`**, and **`feature_importance.png`**.

For a production system, next steps typically include **calibration**, **threshold optimization** using real **costs**, **temporal validation** (fraud drift), and **fairness** review across customer segments.